In [22]:
import numpy as np
import pandas as pd
import json
import os

In [23]:
SEED = 42
np.random.seed(SEED)
N = 2000

In [24]:
WEIGHTS = {
    "tx_regularity_score":   0.25,
    "bill_payment_rate":     0.25,
    "income_stability":      0.20,
    "has_savings_behaviour": 0.10,
    "merchant_diversity":    0.08,
    "avg_monthly_inflow":    0.08,
    "account_age_months":    0.04,
}

In [25]:
tx_regularity_score   = np.random.beta(3, 4, N) * 100
bill_payment_rate     = np.random.beta(3, 3, N) * 100
income_stability      = np.random.beta(2, 4, N) * 100
has_savings_behaviour = np.random.binomial(1, 0.35, N).astype(float)
merchant_diversity    = np.random.poisson(4, N).clip(0, 15).astype(float)
avg_monthly_inflow    = np.random.exponential(35, N).clip(5, 500)
account_age_months    = np.random.randint(1, 49, N).astype(float)

df = pd.DataFrame({
    "tx_regularity_score":   tx_regularity_score,
    "bill_payment_rate":     bill_payment_rate,
    "income_stability":      income_stability,
    "has_savings_behaviour": has_savings_behaviour,
    "merchant_diversity":    merchant_diversity,
    "avg_monthly_inflow":    avg_monthly_inflow,
    "account_age_months":    account_age_months,
})

In [26]:
merchant_norm = (df["merchant_diversity"] / 15) * 100
inflow_norm   = (df["avg_monthly_inflow"] / 500) * 100
age_norm      = (df["account_age_months"] / 48)  * 100

df["credit_score"] = (
    WEIGHTS["tx_regularity_score"]   * df["tx_regularity_score"]  +
    WEIGHTS["bill_payment_rate"]     * df["bill_payment_rate"]     +
    WEIGHTS["income_stability"]      * df["income_stability"]      +
    WEIGHTS["has_savings_behaviour"] * df["has_savings_behaviour"] * 100 +
    WEIGHTS["merchant_diversity"]    * merchant_norm               +
    WEIGHTS["avg_monthly_inflow"]    * inflow_norm                 +
    WEIGHTS["account_age_months"]    * age_norm
).clip(0, 100)

In [27]:
print(f"Nulls          : {df.isnull().sum().sum()}")
print(f"Row count      : {len(df)}")
print(f"\nScore stats:")
print(f"  Mean         : {df['credit_score'].mean():.2f}")
print(f"  Median       : {df['credit_score'].median():.2f}")
print(f"  Std          : {df['credit_score'].std():.2f}")
print(f"  Min / Max    : {df['credit_score'].min():.2f} / {df['credit_score'].max():.2f}")

thin  = (df["credit_score"] < 40).sum()
std   = df["credit_score"].between(40, 59).sum()
prime = (df["credit_score"] >= 60).sum()

print(f"\nTier breakdown:")
print(f"  Thin   < 40  : {thin}  ({thin/N*100:.1f}%)")
print(f"  Standard40–59: {std}   ({std/N*100:.1f}%)")
print(f"  Prime  60+   : {prime}  ({prime/N*100:.1f}%)")
print(f"\nSavings rate   : {df['has_savings_behaviour'].mean()*100:.1f}%")
print(f"Median inflow  : ₦{df['avg_monthly_inflow'].median():.1f}k")

Nulls          : 0
Row count      : 2000

Score stats:
  Mean         : 37.52
  Median       : 37.10
  Std          : 8.80
  Min / Max    : 9.88 / 66.04

Tier breakdown:
  Thin   < 40  : 1252  (62.6%)
  Standard40–59: 731   (36.5%)
  Prime  60+   : 11  (0.5%)

Savings rate   : 31.6%
Median inflow  : ₦23.6k


In [28]:
os.makedirs("../data", exist_ok=True)
df.to_csv("../data/creditpulse_data.csv", index=False)
with open("../data/feature_weights.json", "w") as f:
    json.dump(WEIGHTS, f, indent=2)

print("Saved → data/creditpulse_data.csv")
print("Saved → data/feature_weights.json")

Saved → data/creditpulse_data.csv
Saved → data/feature_weights.json
